## 1. Download and parse the data

***!!!Set your TFL key using an invironment vairable!!!***

on windows: command is `set TFL_API_KEY=your_key_here` in command prompt or `$env:TFL_API_KEY = "your_key_here"` in powershell or set it in windows
get your key at (TFL API registration)[https://api-portal.tfl.gov.uk/]

In [ ]:
!pip install requests pandas openpyxl tqdm networkx plotly nbformat scipy

import os, json, time, math, requests
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

TFL_API_KEY = os.environ.get("TFL_API_KEY", "YOUR_KEY_HERE")

def download(url, dest, chunk=8192):
    """GET url → dest, skipping if already present."""
    if dest.exists():
        print(f"  [skip] {dest.name}")
        return dest
    print(f"  [GET]  {url}")
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True,
                                      desc=dest.name, leave=False) as bar:
        for data in r.iter_content(chunk):
            f.write(data)
            bar.update(len(data))
    print(f"  [OK]   {dest}")
    return dest

We then download the station data before getting their crowding data for each station (this is the step that requires the TfL key)

1. We use Oliver O'Brien's GeoJSON files (derived from OpenStreetMap) which provide every TfL station's coordinates and the ordered stop sequences for each line branch.
2. Annual Station Entry/Exit Counts - We download three years for comparison directly from TfL:
- 2019 — pre-COVID baseline (peak network usage)
- 2022 — COVID recovery year
- 2023 — most recent complete year

In [ ]:
# Structure of TfL stations
BASE = "https://raw.githubusercontent.com/oobrien/vis/master/tubecreature/data"
download(f"{BASE}/tfl_stations.json", DATA_DIR / "tfl_stations.json")
download(f"{BASE}/tfl_lines.json",    DATA_DIR / "tfl_lines.json")

# Download crowding data
AC = "https://crowding.data.tfl.gov.uk/Annual%20Station%20Counts"
annual_urls = {
    "2019": (f"{AC}/2019/AnnualisedEntryExit_2019.xlsx",   "annual_counts_2019.xlsx"),
    "2022": (f"{AC}/2022/AC2022_AnnualisedEntryExit.xlsx", "annual_counts_2022.xlsx"),
    "2023": (f"{AC}/2023/AC2023_AnnualisedEntryExit.xlsx", "annual_counts_2023.xlsx"),
}
for yr, (url, fname) in annual_urls.items():
    download(url, DATA_DIR / fname)

crowding_path = DATA_DIR / "station_crowding.json"

# Check if existing file is valid (non-empty list)
if crowding_path.exists():
    with open(crowding_path) as f:
        _cr = json.load(f)
    if len(_cr) == 0:
        print("  [stale] station_crowding.json is empty — deleting and re-fetching...")
        crowding_path.unlink()

if crowding_path.exists():
    print(f"  [skip] {crowding_path.name}")
elif TFL_API_KEY == "YOUR_KEY_HERE":
    print("Warning: TFL_API_KEY not set — skipping crowding download.")
else:
    TFL_LINES = [
        "bakerloo", "central", "circle", "district", "hammersmith-city",
        "jubilee", "metropolitan", "northern", "piccadilly", "victoria",
        "waterloo-city"
    ]

    print("Fetching station list")
    stops = []
    for line in TFL_LINES:
        r = requests.get(f"https://api.tfl.gov.uk/Line/{line}/StopPoints",
                         params={"app_key": TFL_API_KEY}, timeout=30)
        r.raise_for_status()
        stops.extend(r.json())
        time.sleep(0.1)

    seen, stations = set(), []
    for s in stops:
        nid = s.get("naptanId", "")
        if nid and nid not in seen:
            seen.add(nid)
            stations.append({
                "naptanId": nid,
                "name":     s.get("commonName", ""),
                "lat":      s.get("lat"),
                "lon":      s.get("lon"),
            })
    print(f"  Found {len(stations)} unique stations")

    results = []
    for st in tqdm(stations, desc="Crowding API"):
        r = requests.get(
            f"https://api.tfl.gov.uk/crowding/{st['naptanId']}",
            params={"app_key": TFL_API_KEY}, timeout=15
        )
        crowding = r.json() if r.status_code == 200 else {}
        results.append({**st, "crowding": crowding})
        time.sleep(0.1)

    with open(crowding_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  [OK]   {len(results)} stations → {crowding_path}")

Now we parse the data, we need to flat the geodata into a df (one row per station)

Then we actually construct the graph:
We first collapse the pollylines into "station-to-station" edges.
Approach:
1. Build a lookup: for each station, find the nearest coordinate in the GeoJSON
2. Walk each branch's coordinate list, every time we hit a coordinate that
   snaps to a station, record an edge from the previous snapped station
3. Compute Haversine distance and estimated travel time for each edge
Average tube speed ~33 km/h gives `travel_time_min ≈ dist_km / 33 × 60`

In [ ]:
# Flatten GeoJSON into df
with open(DATA_DIR / "tfl_stations.json") as f:
    stations_geo = json.load(f)

stations_df = pd.DataFrame([
    {
        "station_id": feat["properties"]["id"],
        "name":       feat["properties"]["name"],
        "zone":       feat["properties"].get("zone"),
        "lines":      [l["name"] for l in feat["properties"].get("lines", [])],
        "lon":        feat["geometry"]["coordinates"][0],
        "lat":        feat["geometry"]["coordinates"][1],
    }
    for feat in stations_geo["features"]
])

print(f"stations_df: {stations_df.shape}")
print(stations_df.head(3).to_string())

# Construct edges
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat, dlon = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = (math.sin(dlat/2)**2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2))
         * math.sin(dlon/2)**2)
    return R * 2 * math.asin(math.sqrt(a))

AVG_TUBE_SPEED_KMH = 33.0
SNAP_THRESHOLD_KM  = 0.15

# Build (lat, lon, station_id) lookup
station_coords = stations_df[["station_id", "name", "lat", "lon"]].values

def snap_to_station(lat, lon):
    """Return (station_id, name) of nearest station within threshold, else None."""
    best_id, best_name, best_d = None, None, float("inf")
    for sid, sname, slat, slon in station_coords:
        d = haversine_km(lat, lon, slat, slon)
        if d < best_d:
            best_d, best_id, best_name = d, sid, sname
    if best_d <= SNAP_THRESHOLD_KM:
        return best_id, best_name
    return None, None

with open(DATA_DIR / "tfl_lines.json") as f:
    lines_geo = json.load(f)

edges = []
for feat in lines_geo["features"]:
    props     = feat["properties"]
    line_name = props["lines"][0]["name"] if props.get("lines") else "Unknown"
    coords    = feat["geometry"]["coordinates"]

    prev_sid, prev_name = None, None
    prev_lat, prev_lon  = None, None

    for lon, lat in coords:
        sid, sname = snap_to_station(lat, lon)
        if sid is not None:
            if prev_sid is not None and sid != prev_sid:
                dist = haversine_km(prev_lat, prev_lon, lat, lon)
                edges.append({
                    "line":            line_name,
                    "station_a_id":    prev_sid,
                    "station_a":       prev_name,
                    "station_b_id":    sid,
                    "station_b":       sname,
                    "dist_km":         round(dist, 4),
                    "travel_time_min": round(dist / AVG_TUBE_SPEED_KMH * 60, 2),
                })
            prev_sid, prev_name = sid, sname
            prev_lat, prev_lon  = lat, lon

edges_df = pd.DataFrame(edges).drop_duplicates(
    subset=["line", "station_a_id", "station_b_id"]
)
# Drop self-loops (same station matched under two different IDs)
edges_df = edges_df[edges_df["station_a"] != edges_df["station_b"]].copy()
# Drop implausibly short edges (< 0.2 min)
edges_df = edges_df[edges_df["travel_time_min"] >= 0.2].copy()

print(f"edges_df: {edges_df.shape}")
print(f"Lines:    {sorted(edges_df['line'].unique())}")
print(f"\nTravel time stats (mins):\n{edges_df['travel_time_min'].describe().round(2)}")
print(f"\nFirst 5 edges:")
print(edges_df.head(5).to_string())

Parse our Annual counts into a df and clean up data
Then we standardise the station code column before aligning the days columns as the data from 2019 is in a different format

In [ ]:
DAY_COLS_4 = ["mon", "twt", "fri", "sat"]           # 2019
DAY_COLS_5 = ["mon", "twt", "fri", "sat", "sun"]    # 2022/23

# TfL files use duplicate column headers so we need to renamne them
def rename_annual_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Rename auto-generated entries.N / exits.N columns to meaningful names."""
    cols = list(df.columns)
    entry_idx = [i for i, c in enumerate(cols) if c.startswith("entries")]
    exit_idx  = [i for i, c in enumerate(cols) if c.startswith("exits")]
    enex_idx  = [i for i, c in enumerate(cols) if c.startswith("en_ex")]

    day_cols = DAY_COLS_4 if len(entry_idx) == 4 else DAY_COLS_5

    rename = {}
    for i, idx in enumerate(entry_idx):
        rename[cols[idx]] = f"entries_{day_cols[i]}"
    for i, idx in enumerate(exit_idx):
        rename[cols[idx]] = f"exits_{day_cols[i]}"

    # 2019 has 1 aggregate; 2022/23 have 3
    agg_names = ["en_ex_weekly", "en_ex_12week", "en_ex_annual"]
    if len(enex_idx) == 1:
        rename[cols[enex_idx[0]]] = "en_ex_annual"
    else:
        for i, idx in enumerate(enex_idx):
            rename[cols[idx]] = agg_names[i] if i < len(agg_names) else f"en_ex_{i}"

    df = df.rename(columns=rename)

    # Standardise station-code column name (nlc → mnlc for 2019)
    if "nlc" in df.columns and "mnlc" not in df.columns:
        df = df.rename(columns={"nlc": "mnlc", "asc": "masc"})

    # Cast all numeric columns (2023 loads as object/string)
    num_cols = [c for c in df.columns if c not in
                ["mode", "masc", "station", "coverage", "source"]]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


def parse_annual_counts(path: Path) -> pd.DataFrame:
    xl = pd.ExcelFile(path)
    data_sheet = None
    for s in xl.sheet_names:
        probe = pd.read_excel(path, sheet_name=s, header=None, nrows=10)
        if probe[0].astype(str).str.strip().eq("Mode").any():
            data_sheet = s
            break
    if data_sheet is None:
        raise ValueError(f"No data sheet in {path.name}. Sheets: {xl.sheet_names}")

    raw = pd.read_excel(path, sheet_name=data_sheet, header=None)
    header_row = int(raw[raw[0].astype(str).str.strip() == "Mode"].index[0])

    df = pd.read_excel(path, sheet_name=data_sheet, header=header_row)
    df = df.dropna(how="all").dropna(axis=1, how="all")
    first_col = df.columns[0]
    valid_modes = {"LU", "LO", "DLR", "TfL Rail", "EL"}
    df = df[df[first_col].astype(str).str.strip().isin(valid_modes)].copy()
    df.columns = [str(c).strip().lower().replace(" ", "_").replace("/", "_")
                  for c in df.columns]
    df = rename_annual_cols(df)
    return df

annual_dfs = {}
for yr, (_, fname) in annual_urls.items():
    df = parse_annual_counts(DATA_DIR / fname)
    annual_dfs[yr] = df
    print(f"\nannual_dfs['{yr}']: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    # Show station + first entry col + last col (annualised total), whatever they're called
    entry_cols = [c for c in df.columns if c.startswith("entries_")]
    last_col   = df.columns[-1]
    show_cols  = ["station"] + entry_cols[:2] + [last_col]
    print(df[show_cols].head(3).to_string())

Next, we can parse the crowding JSON data into a df

In [ ]:

crowding_df = pd.DataFrame()

if crowding_path.exists():
    with open(crowding_path) as f:
        cr = json.load(f)

    # Print a raw sample so we can verify the structure
    # sample = next((s for s in cr if s.get("crowding")), None)
    # if sample:
    #     print("Sample crowding entry:")
    #     print(json.dumps(sample, indent=2)[:600])

    rows = []
    for s in cr:
        base = {
            "naptanId": s["naptanId"],
            "name":     s["name"],
            "lat":      s["lat"],
            "lon":      s["lon"],
        }
        crowding = s.get("crowding", {})

        # API returns a dict with a 'timetable' or 'dayOfWeek' list
        # Normalise both possible shapes into a flat list of day entries
        day_entries = crowding.get("daysOfWeek", []) if isinstance(crowding, dict) else []

        if day_entries:
            for day_entry in day_entries:
                day = day_entry.get("dayOfWeek", "")
                for ts in day_entry.get("timeBands", []):
                    rows.append({
                        **base,
                        "day":       day,
                        "time_band": ts.get("timeBand"),
                        "busyness":  ts.get("percentageOfBaseLine"),
                    })
        else:
            # Station exists but no crowding data available
            rows.append({**base, "day": None, "time_band": None, "busyness": None})

    crowding_df = pd.DataFrame(rows)
    n_with = crowding_df["busyness"].notna().sum()
    print(f"\ncrowding_df: {crowding_df.shape} — {n_with} rows with busyness data")
    print(crowding_df[crowding_df["busyness"].notna()].head(5).to_string())
else:
    print("station_crowding.json not found.")

Now that we have the station data, we also want to collect some IMD and LSOA data
Sources:
- IMD 2019:  https://assets.publishing.service.gov.uk/government/uploads/system/uploads/attachment_data/file/833970/File_1_-_IMD2019_Index_of_Multiple_Deprivation.xlsx
- LSOA Population (Census 2021): https://www.nomisweb.co.uk/api/v01/dataset/NM_2002_1.data.csv
- LSOA Centroids: https://github.com/drkane/geo-lookups/blob/master/lsoa_latlong.csv

In [ ]:
import zipfile

# 1. IMD 2019 — Index of Multiple Deprivation decile per LSOA (England)
IMD_URL  = (
    "https://assets.publishing.service.gov.uk/government/uploads/"
    "system/uploads/attachment_data/file/833970/"
    "File_1_-_IMD2019_Index_of_Multiple_Deprivation.xlsx"
)
IMD_PATH = DATA_DIR / "imd2019.xlsx"
download(IMD_URL, IMD_PATH)

# 2. LSOA population (Census 2021 TS001 bulk zip via Nomis — small, reliable)

POP_ZIP_URL  = "https://www.nomisweb.co.uk/output/census/2021/census2021-ts001.zip"
POP_ZIP_PATH = DATA_DIR / "census2021-ts001.zip"
POP_PATH     = DATA_DIR / "lsoa_population.csv"
download(POP_ZIP_URL, POP_ZIP_PATH)

# Extract only the LSOA-level CSV from the zip
with zipfile.ZipFile(POP_ZIP_PATH) as z:
    lsoa_csv = [n for n in z.namelist() if "lsoa" in n.lower() and n.endswith(".csv")]
    if not lsoa_csv:
        raise FileNotFoundError(f"No LSOA CSV found in zip. Contents: {z.namelist()}")
    print(f"  Extracting {lsoa_csv[0]}")
    with z.open(lsoa_csv[0]) as src, open(POP_PATH, "wb") as dst:
        dst.write(src.read())

# 3. LSOA 2021 Centroids — from drkane/geo-lookups (GitHub, no auth required)
# Source: https://github.com/drkane/geo-lookups/blob/master/lsoa_latlong.csv
CENT_PATH = DATA_DIR / "lsoa_centroids.csv"
CENT_URL  = (
    "https://raw.githubusercontent.com/drkane/geo-lookups/master/lsoa_latlong.csv"
)
download(CENT_URL, CENT_PATH)

Now we parse a 3 sets of data

In [ ]:
# Parse IMD (Decile: 1 = most deprived, 10 = least deprived)
imd_raw = pd.read_excel(IMD_PATH, sheet_name="IMD2019", engine="openpyxl")
imd_df  = imd_raw[["LSOA code (2011)", "Index of Multiple Deprivation (IMD) Decile"]].copy()
imd_df.columns = ["lsoa_code", "imd_decile"]
imd_df["lsoa_code"] = imd_df["lsoa_code"].str.strip()
# Convert decile to deprivation weight: decile 1 → 1.0 (most deprived), decile 10 → 0.0
imd_df["imd_score"] = (10 - imd_df["imd_decile"]) / 9.0
print(f"IMD rows: {len(imd_df)}")

# Parse Population
pop_raw = pd.read_csv(POP_PATH)
pop_df  = pop_raw[["geography code", "Residence type: Total; measures: Value"]].copy()
pop_df.columns = ["lsoa_code", "population"]
pop_df = pop_df.dropna(subset=["lsoa_code", "population"])
pop_df["lsoa_code"] = pop_df["lsoa_code"].astype(str).str.strip()
pop_df["population"] = pd.to_numeric(pop_df["population"], errors="coerce")
pop_df = pop_df.dropna(subset=["population"])
print(f"Population rows: {len(pop_df)}")

# Parse Centroids
cent_raw = pd.read_csv(CENT_PATH)
print(f"  Centroid columns: {cent_raw.columns.tolist()}")
# Use 2021 codes where present, fall back to 2011
code_col = "LSOACD"
cent_df  = cent_raw[[code_col, "latitude", "longitude"]].copy()
cent_df.columns = ["lsoa_code", "lat", "lon"]
cent_df = cent_df.dropna()
cent_df["lsoa_code"] = cent_df["lsoa_code"].astype(str).str.strip()
print(f"Centroid rows: {len(cent_df)}")

Join into a single LSOA master table

Note: IMD uses 2011 LSOA codes; Census 2021 uses 2021 codes.
Most inner-London LSOAs are stable; unmatched rows get a neutral weight of 0.5.

In [ ]:
lsoa_df = (
    cent_df
    .merge(pop_df,                             on="lsoa_code", how="inner")
    .merge(imd_df[["lsoa_code", "imd_score"]], on="lsoa_code", how="left")
)

# Keep only Greater London bounding box
lsoa_df = lsoa_df[
    (lsoa_df["lat"].between(51.28, 51.72)) &
    (lsoa_df["lon"].between(-0.55,  0.35))
].copy().reset_index(drop=True)

# Unmatched LSOAs (2011→2021 code mismatch) get neutral deprivation weight
lsoa_df["deprivation_weight"] = lsoa_df["imd_score"].fillna(0.5)

n_matched = lsoa_df["imd_score"].notna().sum()
print(f"\nlsoa_df (London): {lsoa_df.shape}")
print(f"LSOAs matched to IMD: {n_matched} / {len(lsoa_df)}")
print(lsoa_df[["lsoa_code", "lat", "lon", "population", "imd_score", "deprivation_weight"]].head(5).to_string())

# Pre-build numpy arrays for fast spatial lookups in the solver
lsoa_coords_arr = lsoa_df[["lat", "lon"]].to_numpy()
lsoa_pop_arr    = lsoa_df["population"].to_numpy()
lsoa_dep_arr    = lsoa_df["deprivation_weight"].to_numpy()

## 2. Compute Node Weights and Build Graph

Compute Node Weights
Merge 2023 annual ridership onto stations_df. Where a station appears under multiple modes (e.g. Bank in both LU and DLR) we sum across all modes.

In [ ]:
ridership = (
    annual_dfs["2023"]
    .groupby("station")["en_ex_annual"]
    .sum()
    .reset_index()
    .rename(columns={"station": "name", "en_ex_annual": "ridership_2023"})
)

# Normalise names for joining
ridership["name_key"]   = ridership["name"].str.strip().str.lower()
stations_df["name_key"] = stations_df["name"].str.strip().str.lower()

stations_df = stations_df.merge(
    ridership[["name_key", "ridership_2023"]], on="name_key", how="left"
)

# Fill missing ridership with 0
stations_df["ridership_2023"] = stations_df["ridership_2023"].fillna(0)

matched = (stations_df["ridership_2023"] > 0).sum()
print(f"Stations matched to ridership data: {matched} / {len(stations_df)}")
print(stations_df[["name", "ridership_2023"]].nlargest(5, "ridership_2023").to_string())

Now we actually build the graph
Each node stores coordinates and ridership; each edge stores line and travel time.

In [ ]:
import networkx as nx

G = nx.MultiGraph()

for _, row in stations_df.iterrows():
    G.add_node(
        row["station_id"],
        name=row["name"],
        lat=row["lat"],
        lon=row["lon"],
        zone=str(row.get("zone", "?")),
        ridership=float(row["ridership_2023"]),
    )

for _, row in edges_df.iterrows():
    if row["station_a_id"] in G and row["station_b_id"] in G:
        G.add_edge(
            row["station_a_id"],
            row["station_b_id"],
            line=row["line"],
            travel_time_min=row["travel_time_min"],
            dist_km=row["dist_km"],
        )

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Connected components: {nx.number_connected_components(G)}")

# Top 5 nodes by ridership
top5 = sorted(G.nodes(data=True), key=lambda x: x[1]["ridership"], reverse=True)[:5]
print("\nTop 5 stations by ridership:")
for sid, d in top5:
    print(f"  {d['name']:35s}  {d['ridership']:>12,.0f}")

We will make a interactive map visualisation to make sure everything has worked as expected

Edges are coloured by line, thickness proportional to 1/travel_time so shorter (busier) links appear thicker
Nodes are sized and coloured by 2023 ridership

In [ ]:
import plotly.graph_objects as go

LINE_COLOURS = {
    "Bakerloo":             "#B36305",
    "Central":              "#E32017",
    "Circle":               "#FFD300",
    "District":             "#00782A",
    "Hammersmith & City":   "#F3A9BB",
    "Jubilee":              "#A0A5A9",
    "Metropolitan":         "#9B0056",
    "Northern":             "#000000",
    "Piccadilly":           "#003688",
    "Victoria":             "#0098D4",
    "Waterloo & City":      "#95CDBA",
    "DLR":                  "#00A4A7",
    "Elizabeth line":       "#6950A1",
    "London Overground":    "#EE7C0E",
    "Tramlink":             "#84B817",
    "Crossrail 2":          "#AB008B",
    "Thameslink 6tph line": "#D42E82",
    "IFS Cloud Cable Car":  "#E21836",
    "East London":          "#EE7C0E",
}

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Connected components: {nx.number_connected_components(G)}")

# Top 5 nodes by ridership
top5 = sorted(G.nodes(data=True), key=lambda x: x[1]["ridership"], reverse=True)[:5]
print("\nTop 5 stations by ridership:")
for sid, d in top5:
    print(f"  {d['name']:35s}  {d['ridership']:>12,.0f}")

fig = go.Figure()

# ── Edges — one Scattermap trace per line ────────────────────────────────────
for line_name, colour in LINE_COLOURS.items():
    line_edges = edges_df[edges_df["line"] == line_name]
    if line_edges.empty:
        continue

    lats, lons, texts = [], [], []
    for _, e in line_edges.iterrows():
        a = G.nodes.get(e["station_a_id"])
        b = G.nodes.get(e["station_b_id"])
        if not a or not b:
            continue
        lats += [a["lat"], b["lat"], None]
        lons += [a["lon"], b["lon"], None]
        hover = (f"{e['station_a']} → {e['station_b']}<br>"
                 f"Line: {line_name}<br>"
                 f"Distance: {e['dist_km']:.2f} km<br>"
                 f"Travel time: {e['travel_time_min']:.1f} min")
        texts += [hover, hover, None]

    fig.add_trace(go.Scattermap(
        lat=lats, lon=lons,
        mode="lines",
        line=dict(color=colour, width=2),
        hovertext=texts, hoverinfo="text",
        name=line_name,
    ))

# ── Nodes ─────────────────────────────────────────────────────────────────────
node_data  = list(G.nodes(data=True))
node_lats  = [d["lat"]       for _, d in node_data]
node_lons  = [d["lon"]       for _, d in node_data]
node_rid   = [d["ridership"] for _, d in node_data]
node_hover = [
    f"<b>{d['name']}</b><br>Zone: {d['zone']}<br>"
    f"2023 ridership: {d['ridership']:,.0f}<br>"
    f"Connections: {G.degree(sid)}"
    for sid, d in node_data
]

max_rid   = max(r for r in node_rid if r > 0)
node_size = [5 + 20 * (r / max_rid) ** 0.4 for r in node_rid]

fig.add_trace(go.Scattermap(
    lat=node_lats, lon=node_lons,
    mode="markers",
    marker=dict(
        size=node_size,
        color=node_rid,
        colorscale="YlOrRd",
        colorbar=dict(title="2023 Annual<br>Ridership", thickness=15),
        cmin=0, cmax=max_rid,
        opacity=0.9,
    ),
    hovertext=node_hover,
    hoverinfo="text",
    name="Stations",
))

fig.update_layout(
    title="London Underground — Node & Edge Weights",
    map=dict(
        style="carto-positron",
        center=dict(lat=51.509, lon=-0.118),
        zoom=10,
    ),
    legend=dict(x=0, y=1, bgcolor="rgba(255,255,255,0.8)", font=dict(size=10)),
    height=800,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

## 3. Solve the graph

### 3.1 Proof of concept solver

First we will make a scoring  
we use a simple `score = travel_time_min × demand_pressure` system as a proof of concept  
`travel_time_min` = current edge weight (time saved by splitting the edge)  
`demand_pressure` = mean of A and B's ridership ÷ their graph degree (high ridership + few connections = overloaded station)  

then we solve by simply scoring the candadates

In [ ]:
LU_LINES = {
    "Bakerloo", "Central", "Circle", "District", "Hammersmith & City",
    "Jubilee", "Metropolitan", "Northern", "Piccadilly", "Victoria",
    "Waterloo & City"
}

lu_edges = edges_df[edges_df["line"].isin(LU_LINES)].copy()
print(f"Underground edges: {len(lu_edges)}")

# Build a pressure score for each node: pressure = ridership / max(degree, 1)

node_pressure = {}
for sid, data in G.nodes(data=True):
    ridership = data.get("ridership", 0)
    degree    = max(G.degree(sid), 1)
    node_pressure[sid] = ridership / degree

# Score each edge
scores = []
for _, row in lu_edges.iterrows():
    a_id = row["station_a_id"]
    b_id = row["station_b_id"]

    p_a = node_pressure.get(a_id, 0)
    p_b = node_pressure.get(b_id, 0)
    mean_pressure = (p_a + p_b) / 2

    score = row["travel_time_min"] * mean_pressure

    # Candidate location = geographic midpoint of the edge
    a_data = G.nodes.get(a_id, {})
    b_data = G.nodes.get(b_id, {})
    mid_lat = (a_data.get("lat", 0) + b_data.get("lat", 0)) / 2
    mid_lon = (a_data.get("lon", 0) + b_data.get("lon", 0)) / 2

    scores.append({
        "line":            row["line"],
        "station_a":       row["station_a"],
        "station_b":       row["station_b"],
        "station_a_id":    a_id,
        "station_b_id":    b_id,
        "dist_km":         row["dist_km"],
        "travel_time_min": row["travel_time_min"],
        "pressure_a":      p_a,
        "pressure_b":      p_b,
        "mean_pressure":   mean_pressure,
        "score":           score,
        "mid_lat":         mid_lat,
        "mid_lon":         mid_lon,
    })

scores_df = pd.DataFrame(scores).sort_values("score", ascending=False).reset_index(drop=True)

print("\nTop 10 candidate locations:")
print(scores_df[["line","station_a","station_b","dist_km",
                  "travel_time_min","mean_pressure","score"]].head(10).to_string())

Shows are top 3 candadites according to our socring criteria on a map with colour coded edges

In [ ]:


# Normalise scores 0 to 1 for colour mapping
s_min, s_max = scores_df["score"].min(), scores_df["score"].max()
scores_df["score_norm"] = (scores_df["score"] - s_min) / (s_max - s_min)

def score_to_colour(norm: float) -> str:
    """Map 0–1 score to a blue→yellow→red colour."""
    r = int(min(255, norm * 2 * 255))
    g = int(max(0, 255 - abs(norm - 0.5) * 2 * 255))
    b = int(max(0, (1 - norm * 2) * 255))
    return f"rgb({r},{g},{b})"

fig_heat = go.Figure()

# Draw all scored edges
for _, row in scores_df.iterrows():
    a = G.nodes.get(row["station_a_id"], {})
    b = G.nodes.get(row["station_b_id"], {})
    if not a or not b:
        continue

    colour = score_to_colour(row["score_norm"])
    width  = 2 + row["score_norm"] * 6   # thicker = higher score

    fig_heat.add_trace(go.Scattermap(
        lat=[a["lat"], b["lat"]],
        lon=[a["lon"], b["lon"]],
        mode="lines",
        line=dict(color=colour, width=width),
        hovertext=(
            f"<b>{row['station_a']} → {row['station_b']}</b><br>"
            f"Line: {row['line']}<br>"
            f"Distance: {row['dist_km']:.2f} km<br>"
            f"Travel time: {row['travel_time_min']:.1f} min<br>"
            f"Mean pressure: {row['mean_pressure']:,.0f}<br>"
            f"Score: {row['score']:,.0f}"
        ),
        hoverinfo="text",
        showlegend=False,
    ))

# Overlay all stations as small grey dots
fig_heat.add_trace(go.Scattermap(
    lat=[d["lat"] for _, d in G.nodes(data=True)],
    lon=[d["lon"] for _, d in G.nodes(data=True)],
    mode="markers",
    marker=dict(size=5, color="grey", opacity=0.5),
    hovertext=[d["name"] for _, d in G.nodes(data=True)],
    hoverinfo="text",
    showlegend=False,
))

# Mark top 3 candidates
top3 = scores_df.head(3)
fig_heat.add_trace(go.Scattermap(
    lat=top3["mid_lat"].tolist(),
    lon=top3["mid_lon"].tolist(),
    mode="markers+text",
    marker=dict(size=18, color="gold", symbol="star", opacity=1.0),
    text=[f"#{i+1}" for i in range(3)],
    textposition="top right",
    textfont=dict(size=14, color="black"),
    hovertext=[
        f"<b>Candidate #{i+1}</b><br>"
        f"{row['station_a']} → {row['station_b']}<br>"
        f"Line: {row['line']}<br>"
        f"Score: {row['score']:,.0f}"
        for i, (_, row) in enumerate(top3.iterrows())
    ],
    hoverinfo="text",
    name="Top 3 candidates",
    showlegend=True,
))

# Dummy colourbar trace
fig_heat.add_trace(go.Scattermap(
    lat=[None], lon=[None],
    mode="markers",
    marker=dict(
        size=0,
        color=[0, 1],
        colorscale=[[0, "blue"], [0.5, "yellow"], [1, "red"]],
        colorbar=dict(
            title="Edge Score<br>(higher = better<br>candidate)",
            thickness=15,
            tickvals=[0, 1],
            ticktext=["Low", "High"],
        ),
        showscale=True,
    ),
    showlegend=False,
))

fig_heat.update_layout(
    title="New Station Solver — Edge Score Heatmap (Underground only)",
    map=dict(
        style="carto-positron",
        center=dict(lat=51.509, lon=-0.118),
        zoom=10,
    ),
    legend=dict(x=0, y=1, bgcolor="rgba(255,255,255,0.8)"),
    height=800,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig_heat.show()

### 3.2 Improved solver

#### 3.2.1 Parameters

$Score = \alpha \cdot \Delta\text{Travel\_time} + \beta \cdot \text{net\_new\_demand} + \gamma \cdot \text{deprivation\_weight}$

$\Delta\text{ATT\_network}$ – gravity-weighted reduction in average travel time across network  
$\text{net\_new\_demand}$ – population in new station's catchment area not served by other existing stations  
$\text{deprivation\_wt}$ – IMD-weighted deprivation of the catchment population

In [ ]:
# ================= Tunable parameters =================
ALPHA        = 0.5   # weight: network travel-time benefit
BETA         = 0.3   # weight: net new demand
GAMMA        = 0.2   # weight: equity / deprivation

CATCHMENT_KM    = 0.8   # walk-catchment radius for the new station
EXISTING_CAP_KM = 0.6   # radius within which existing stations are "competitors"
SAMPLE_FRAC     = 0.25  # fraction of stations sampled for ΔATT (speed/accuracy trade-off)
# ======================================================

LU_LINES = {
    "Bakerloo", "Central", "Circle", "District", "Hammersmith & City",
    "Jubilee", "Metropolitan", "Northern", "Piccadilly", "Victoria",
    "Waterloo & City"
}

lu_edges = edges_df[edges_df["line"].isin(LU_LINES)].copy()
print(f"Underground edges to score: {len(lu_edges)}")

#### 3.2.2 Build baseline weighted graph for shortest-path queries

In [ ]:
def build_simple_graph(extra_node=None, extra_edges=None):
    """
    Collapse MultiGraph to a simple weighted Graph (min travel time per pair).
    Optionally inject a new candidate node + edges.
    """
    Gs = nx.Graph()
    for sid, d in G.nodes(data=True):
        Gs.add_node(sid, **d)
    for _, row in edges_df.iterrows():
        a, b, t = row["station_a_id"], row["station_b_id"], row["travel_time_min"]
        if Gs.has_edge(a, b):
            if Gs[a][b]["weight"] > t:
                Gs[a][b]["weight"] = t
        else:
            Gs.add_edge(a, b, weight=t)
    if extra_node:
        Gs.add_node(extra_node["id"], **extra_node)
    if extra_edges:
        for e in extra_edges:
            Gs.add_edge(e["station_a_id"], e["station_b_id"], weight=e["travel_time_min"])
    return Gs

G_base = build_simple_graph()

#### 3.2.3 Sample $\Delta\text{ATT\_network}$
Now we sample station pairs for our $\Delta\text{ATT\_network}$ calculation using a $flow(i,j) \propto i_ridership * j_ridership$ method

In [ ]:
rng         = np.random.default_rng(42)
all_sids    = [s for s in G_base.nodes() if G_base.nodes[s].get("ridership", 0) > 0]
n_sample    = max(30, int(len(all_sids) * SAMPLE_FRAC))
base_sample = set(rng.choice(all_sids, size=min(n_sample, len(all_sids)),
                              replace=False).tolist())

# Guarantee all LU edge endpoints are in the sample
lu_endpoint_sids = set(lu_edges["station_a_id"]).union(set(lu_edges["station_b_id"]))
lu_endpoint_sids = {s for s in lu_endpoint_sids
                    if G_base.nodes.get(s, {}).get("ridership", 0) > 0}

sampled_sids = list(base_sample | lu_endpoint_sids)

# Gravity flow matrix
rid_vec  = np.array([G_base.nodes[s]["ridership"] for s in sampled_sids])
flow_mat = np.outer(rid_vec, rid_vec)
np.fill_diagonal(flow_mat, 0)

print(f"Sampled {len(sampled_sids)} stations for ΔATT "
      f"({len(sampled_sids)**2:,} pairs) "
      f"[{len(lu_endpoint_sids)} LU endpoints guaranteed]")

def compute_att(Gs, sids, flow):
    """Gravity-weighted average travel time across sampled pairs."""
    lengths = dict(nx.all_pairs_dijkstra_path_length(Gs, weight="weight"))
    total_flow, total_wtt = 0.0, 0.0
    for i, a in enumerate(sids):
        for j, b in enumerate(sids):
            if i >= j:
                continue
            f  = flow[i, j]
            tt = lengths.get(a, {}).get(b, np.nan)
            if np.isfinite(tt) and f > 0:
                total_wtt  += f * tt
                total_flow += f
    return total_wtt / total_flow if total_flow > 0 else 0.0

print("Computing baseline ATT…")
baseline_att = compute_att(G_base, sampled_sids, flow_mat)
print(f"  Baseline ATT = {baseline_att:.3f} min")

#### 3.2.4 LOSA component
Now we perform our lsoa calculation to calculate the new catchment

In [ ]:
def lsoa_catchment(lat, lon, radius_km):
    """Indices into lsoa_df for LSOAs within radius_km of (lat, lon)."""
    dlat = (lsoa_coords_arr[:, 0] - lat) * 111.32
    dlon = (lsoa_coords_arr[:, 1] - lon) * 111.32 * math.cos(math.radians(lat))
    return np.where(np.sqrt(dlat**2 + dlon**2) <= radius_km)[0]

def existing_served_set(mid_lat, mid_lon):
    """
    LSOA indices already served by any existing station within EXISTING_CAP_KM
    of the candidate midpoint. Cached per candidate for speed.
    """
    served = set()
    for sid, d in G.nodes(data=True):
        if haversine_km(mid_lat, mid_lon, d["lat"], d["lon"]) <= EXISTING_CAP_KM:
            for idx in lsoa_catchment(d["lat"], d["lon"], CATCHMENT_KM).tolist():
                served.add(idx)
    return served

#### 3.2.5 Main scoring loop

We finally combine all of this in our main scoring loop to build a list of candadites that we then normalize into weighted scores

In [ ]:
NEW_SID = "__new__"
scores2 = []

print(f"\nScoring {len(lu_edges)} candidate edges…")
for _, row in tqdm(lu_edges.iterrows(), total=len(lu_edges), desc="Candidates"):
    a_id, b_id = row["station_a_id"], row["station_b_id"]
    a_d,  b_d  = G.nodes.get(a_id, {}), G.nodes.get(b_id, {})
    if not a_d or not b_d:
        continue

    mid_lat = (a_d["lat"] + b_d["lat"]) / 2
    mid_lon = (a_d["lon"] + b_d["lon"]) / 2
    half_t  = row["travel_time_min"] / 2

    # delta ATT
    new_node  = {"id": NEW_SID, "lat": mid_lat, "lon": mid_lon, "ridership": 0}
    new_edges = [
        {"station_a_id": a_id,   "station_b_id": NEW_SID, "travel_time_min": half_t},
        {"station_a_id": NEW_SID,"station_b_id": b_id,    "travel_time_min": half_t},
    ]
    G_cand = build_simple_graph(extra_node=new_node, extra_edges=new_edges)
    if G_cand.has_edge(a_id, b_id):
        G_cand.remove_edge(a_id, b_id)

    delta_att = baseline_att - compute_att(G_cand, sampled_sids, flow_mat)

    # Net new demand
    new_idx    = lsoa_catchment(mid_lat, mid_lon, CATCHMENT_KM)
    served_set = existing_served_set(mid_lat, mid_lon)
    new_only   = np.array([i for i in new_idx if i not in served_set])
    net_new_pop = lsoa_pop_arr[new_only].sum() if len(new_only) > 0 else 0.0

    # Equity weight
    if len(new_idx) > 0:
        catch_pop = lsoa_pop_arr[new_idx]
        dep_wt    = (np.dot(catch_pop, lsoa_dep_arr[new_idx]) / catch_pop.sum()
                     if catch_pop.sum() > 0 else 0.0)
    else:
        dep_wt = 0.0

    scores2.append({
        "line":            row["line"],
        "station_a":       row["station_a"],
        "station_b":       row["station_b"],
        "station_a_id":    a_id,
        "station_b_id":    b_id,
        "dist_km":         row["dist_km"],
        "travel_time_min": row["travel_time_min"],
        "mid_lat":         mid_lat,
        "mid_lon":         mid_lon,
        "delta_att":       delta_att,
        "net_new_pop":     net_new_pop,
        "deprivation_wt":  dep_wt,
    })

scores2_df = pd.DataFrame(scores2)

# Normalise components to weighted scores
for col in ["delta_att", "net_new_pop", "deprivation_wt"]:
    lo, hi = scores2_df[col].min(), scores2_df[col].max()
    scores2_df[f"{col}_norm"] = (scores2_df[col] - lo) / (hi - lo) if hi > lo else 0.0

scores2_df["score"] = (
    ALPHA * scores2_df["delta_att_norm"]
  + BETA  * scores2_df["net_new_pop_norm"]
  + GAMMA * scores2_df["deprivation_wt_norm"]
)
scores2_df = scores2_df.sort_values("score", ascending=False).reset_index(drop=True)

print("\nTop 10 candidates (refined solver):")
print(scores2_df[[
    "line", "station_a", "station_b",
    "dist_km", "delta_att", "net_new_pop", "deprivation_wt", "score"
]].head(10).to_string())

Best-per-axis summary

In [ ]:
print("\n── Best candidate by individual axis ──")
for axis, label in [
    ("delta_att",     "Network ΔTT"),
    ("net_new_pop",   "Net new demand"),
    ("deprivation_wt","Equity / deprivation"),
    ("score",         "Overall (weighted)"),
]:
    r = scores2_df.loc[scores2_df[axis].idxmax()]
    print(f"  {label:25s}: {r['station_a']} → {r['station_b']} "
          f"({r['line']})  score={r['score']:.3f}")

#### 3.2.6 Visualise Results of improved solver

In [ ]:
s_min, s_max = scores2_df["score"].min(), scores2_df["score"].max()
scores2_df["score_norm"] = (scores2_df["score"] - s_min) / (s_max - s_min)

fig2 = go.Figure()

# Scored edges
for _, row in scores2_df.iterrows():
    a = G.nodes.get(row["station_a_id"], {})
    b = G.nodes.get(row["station_b_id"], {})
    if not a or not b:
        continue

    norm   = row["score_norm"]
    r_val  = int(min(255, norm * 2 * 255))
    g_val  = int(max(0, 255 - abs(norm - 0.5) * 2 * 255))
    b_val  = int(max(0, (1 - norm * 2) * 255))
    colour = f"rgb({r_val},{g_val},{b_val})"
    width  = 2 + norm * 6

    fig2.add_trace(go.Scattermap(
        lat=[a["lat"], b["lat"]],
        lon=[a["lon"], b["lon"]],
        mode="lines",
        line=dict(color=colour, width=width),
        hovertext=(
            f"<b>{row['station_a']} → {row['station_b']}</b><br>"
            f"Line: {row['line']}<br>"
            f"Distance: {row['dist_km']:.2f} km<br>"
            f"ΔATT: {row['delta_att']:.3f} min<br>"
            f"Net new population: {row['net_new_pop']:,.0f}<br>"
            f"Deprivation weight: {row['deprivation_wt']:.3f}<br>"
            f"Score: {row['score']:.3f}"
        ),
        hoverinfo="text",
        showlegend=False,
    ))

# Existing stations marked as grey dots
fig2.add_trace(go.Scattermap(
    lat=[d["lat"] for _, d in G.nodes(data=True)],
    lon=[d["lon"] for _, d in G.nodes(data=True)],
    mode="markers",
    marker=dict(size=5, color="grey", opacity=0.5),
    hovertext=[d["name"] for _, d in G.nodes(data=True)],
    hoverinfo="text",
    showlegend=False,
))

# Top 5 candidates marked as gold stars
top5 = scores2_df.head(5)
fig2.add_trace(go.Scattermap(
    lat=top5["mid_lat"].tolist(),
    lon=top5["mid_lon"].tolist(),
    mode="markers+text",
    marker=dict(size=18, color="gold", symbol="star", opacity=1.0),
    text=[f"#{i+1}" for i in range(5)],
    textposition="top right",
    textfont=dict(size=13, color="black"),
    hovertext=[
        f"<b>Candidate #{i+1}</b><br>"
        f"{row['station_a']} → {row['station_b']}<br>"
        f"Line: {row['line']}<br>"
        f"ΔATT: {row['delta_att']:.3f} min<br>"
        f"Net new pop: {row['net_new_pop']:,.0f}<br>"
        f"Deprivation: {row['deprivation_wt']:.3f}<br>"
        f"Score: {row['score']:.3f}"
        for i, (_, row) in enumerate(top5.iterrows())
    ],
    hoverinfo="text",
    name="Top 5 candidates",
    showlegend=True,
))

# Best by each individual axis marked as diamonds
for axis, label, colour in [
    ("delta_att",      "Best ΔTT",     "cyan"),
    ("net_new_pop",    "Best demand",  "lime"),
    ("deprivation_wt", "Best equity",  "magenta"),
]:
    r = scores2_df.loc[scores2_df[axis].idxmax()]
    fig2.add_trace(go.Scattermap(
        lat=[r["mid_lat"]],
        lon=[r["mid_lon"]],
        mode="markers+text",
        marker=dict(size=16, color=colour, symbol="diamond", opacity=1.0),
        text=[label],
        textposition="bottom right",
        textfont=dict(size=11, color="black"),
        hovertext=(
            f"<b>{label}</b><br>"
            f"{r['station_a']} → {r['station_b']}<br>"
            f"Line: {r['line']}<br>"
            f"Score: {r['score']:.3f}"
        ),
        hoverinfo="text",
        name=label,
        showlegend=True,
    ))

# Dummy colourbar trace
fig2.add_trace(go.Scattermap(
    lat=[None], lon=[None],
    mode="markers",
    marker=dict(
        size=0,
        color=[0, 1],
        colorscale=[[0, "blue"], [0.5, "yellow"], [1, "red"]],
        colorbar=dict(
            title="Edge Score<br>(higher = better<br>candidate)",
            thickness=15,
            tickvals=[0, 1],
            ticktext=["Low", "High"],
        ),
        showscale=True,
    ),
    showlegend=False,
))

fig2.update_layout(
    title="New Station Solver — Refined Score Heatmap (α*ΔATT + β*demand + γ*equity)",
    map=dict(
        style="carto-positron",
        center=dict(lat=51.509, lon=-0.118),
        zoom=10,
    ),
    legend=dict(x=0, y=1, bgcolor="rgba(255,255,255,0.8)"),
    height=800,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig2.show()

## 3.3 Sensitivity Analysis and Hyperparameter Tuning

We test how sensitive our top candidate is to the choice of scoring weights, Weights are allowed to exceed 1.0 (up to 5.0) to model extreme prioritisation of a single component, this stress-tests whether our chosen candidate is robust or fragile. For each weight combination we record which edge wins, allowing us to identify a candidate that scores well across a wide range of priorities, this is colour-coded on our graphs  
Note: The staircasing of the edges is just an artifact of the low resolution of the sampling not a statstical phenomna

Method:
1. The ternary sensitivity heatmap is rasterised to a pixel image.
2. The ternary space is projected to 2D Cartesian, a grid is sampled
3. using nearest-neighbour interpolation, and rendered as a go.Image.
4. Winning candidate's colour is assigned

In [ ]:
import itertools
from collections import Counter
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
from scipy.spatial import ConvexHull, Delaunay

# Extended grid: each weight in [0.0, 5.0] in steps of 0.05, sum = 1
step   = 0.05
w_vals = [round(x * step, 10) for x in range(int(5.0 / step) + 1)]

combos = [
    (a, b, g)
    for a, b, g in itertools.product(w_vals, repeat=3)
    if abs(a + b + g - 1.0) < 1e-9
]
print(f"Testing {len(combos)} weight combinations…")

# Pre-extract normalised component arrays
att_n  = scores2_df["delta_att_norm"].values
dem_n  = scores2_df["net_new_pop_norm"].values
dep_n  = scores2_df["deprivation_wt_norm"].values
edge_labels = (
    scores2_df["station_a"] + " → " + scores2_df["station_b"]
    + " (" + scores2_df["line"] + ")"
).values

winner_counts = Counter()
results = []

for w_att, w_dem, w_dep in combos:
    s          = w_att * att_n + w_dem * dem_n + w_dep * dep_n
    winner_idx = int(np.argmax(s))
    winner     = edge_labels[winner_idx]
    winner_counts[winner] += 1
    results.append({
        "w_travel_time": w_att,
        "w_demand":      w_dem,
        "w_equity":      w_dep,
        "winner":        winner,
        "winner_score":  float(s[winner_idx]),
    })

results_df = pd.DataFrame(results)

print("\nWinner frequency across all weight combinations:")
for edge, count in winner_counts.most_common(10):
    pct = 100 * count / len(combos)
    print(f"  {pct:5.1f}%  ({count:3d}/{len(combos)})  {edge}")

# Assign integer ID to each unique winner for interpolation
top_winners  = [w for w, _ in winner_counts.most_common()]
winner_to_id = {w: i for i, w in enumerate(top_winners)}
palette = [
    (227, 32,  23),   # red
    (0,   55, 136),   # blue
    (0,  120,  42),   # green
    (155,  0,  86),   # purple
    (179, 99,   5),   # brown
    (0,  152, 212),   # light blue
    (160,165, 169),   # grey
    (255,211,  0),    # yellow
]

# Project ternary to 2D Cartesian
def ternary_to_cart(a, b, c):
    s = a + b + c
    x = 0.5 * (2 * b + c) / s
    y = (np.sqrt(3) / 2) * c / s
    return x, y

# Build scattered (x, y) to winner_id
xs, ys, ids = [], [], []
hover_texts  = []
for _, row in results_df.iterrows():
    wa, wb, wc = row["w_travel_time"], row["w_demand"], row["w_equity"]
    x, y = ternary_to_cart(wa, wb, wc)
    xs.append(x); ys.append(y)
    ids.append(winner_to_id[row["winner"]])
    hover_texts.append(
        f"Travel time weight: {wa:.1f}<br>"
        f"Demand weight: {wb:.1f}<br>"
        f"Equity weight: {wc:.1f}<br>"
        f"<b>Winner: {row['winner']}</b><br>"
        f"Score: {row['winner_score']:.3f}"
    )

xs, ys = np.array(xs), np.array(ys)
pts = np.column_stack([xs, ys])

# Dense 500×500 pixel grid — spans the full extended ternary space
# With weights up to 5
# But we keep the image square covering the bounding box of the projected points
RES   = 2000
x_min, x_max = min(xs) - 0.05, max(xs) + 0.05
y_min, y_max = min(ys) - 0.05, max(ys) + 0.05
xi = np.linspace(x_min, x_max, RES)
yi = np.linspace(y_min, y_max, RES)
XX, YY = np.meshgrid(xi, yi)

ids = np.array(ids)

# Linear interpolation inside convex hull, nearest outside (no extrapolation gaps)
linear_interp  = LinearNDInterpolator(pts, ids.astype(float))
nearest_interp = NearestNDInterpolator(pts, ids)
ZID_lin = linear_interp(XX, YY)
ZID     = np.where(np.isnan(ZID_lin), nearest_interp(XX, YY), ZID_lin)
ZID     = np.round(ZID).astype(int)

# Mask: only show pixels that are inside the CONVEX HULL of the projected points
# (i.e. the true extended ternary region), not the standard unit triangle
hull    = Delaunay(pts)
flat_pts = np.column_stack([XX.ravel(), YY.ravel()])
in_hull  = hull.find_simplex(flat_pts) >= 0
mask     = ~in_hull.reshape(RES, RES)

# Map winner IDs → RGB colours
pal_arr  = np.array(palette + [(200,200,200)] * 20, dtype=np.uint8)
ZID_safe = np.clip(ZID, 0, len(pal_arr) - 1)

img = np.zeros((RES, RES, 4), dtype=np.uint8)
img[:, :, :3] = pal_arr[ZID_safe]
img[:, :,  3] = np.where(mask, 0, 220).astype(np.uint8)
img = img[::-1]

h = np.sqrt(3) / 2

fig_tern = go.Figure()

# Rasterised heatmap — coordinates match the dynamic grid
fig_tern.add_trace(go.Image(
    z=img,
    x0=x_min, dx=(x_max - x_min) / RES,
    y0=y_min, dy=(y_max - y_min) / RES,
    hoverinfo="skip",
))

# Invisible scatter for hover interactivity
fig_tern.add_trace(go.Scatter(
    x=xs, y=ys,
    mode="markers",
    marker=dict(size=8, color="rgba(0,0,0,0)"),
    text=hover_texts,
    hoverinfo="text",
    showlegend=False,
))

# Chosen operating point star
ax, ay = ternary_to_cart(ALPHA, BETA, GAMMA)
fig_tern.add_trace(go.Scatter(
    x=[ax], y=[ay],
    mode="markers",
    marker=dict(color="white", size=18, symbol="star",
                line=dict(color="black", width=2)),
    text=[f"Chosen weights<br>Travel time: {ALPHA}<br>"
          f"Demand: {BETA}<br>Equity: {GAMMA}"],
    hoverinfo="text",
    name="Chosen weights ★",
))

# Triangle border
fig_tern.add_trace(go.Scatter(
    x=[0, 1, 0.5, 0], y=[0, 0, h, 0],
    mode="lines",
    line=dict(color="black", width=1.5),
    hoverinfo="skip",
    showlegend=False,
))

# Corner labels
h = np.sqrt(3) / 2
for label, x, y, anchor in [
    ("w(travel time) = 5.0", 0.0,  -0.04, "right"),
    ("w(demand) = 5.0",      1.0,  -0.04, "left"),
    ("w(equity) = 5.0",      0.5,   h + 0.03, "center"),
]:
    fig_tern.add_annotation(
        x=x, y=y, text=label,
        showarrow=False,
        font=dict(size=11, color="black"),
        bgcolor="white",
        bordercolor="black",
        borderwidth=1,
        borderpad=3,
        xanchor=anchor,
    )

# Discrete legend entries for winners
for i, (winner, _) in enumerate(winner_counts.most_common(len(top_winners))):
    if i >= len(palette):
        break
    r, g, b = palette[i]
    short = winner.split("(")[0].strip()
    fig_tern.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(size=12, color=f"rgb({r},{g},{b})", symbol="square"),
        name=short,
        showlegend=True,
    ))

fig_tern.update_layout(
    title=(
        "Sensitivity Analysis — Winner profile heatmap<br>"
        "<sup>Colour = winning candidate  |  ★ = chosen weights  "
        "|  weights sum to 1, range [−4, 5]</sup>"
    ),
    xaxis=dict(visible=False, range=[x_min - 0.05, x_max + 0.05]),
    yaxis=dict(visible=False, scaleanchor="x",
               range=[y_min - 0.05, y_max + 0.05]),
    height=620,
    plot_bgcolor="white",
    margin=dict(l=20, r=160, t=80, b=20),
    legend=dict(x=1.02, y=1, bgcolor="rgba(255,255,255,0.8)"),
)
fig_tern.show()

Distribution of scores in each component across all candadate edges is plotted  
It is noted that time travel reduction has a very extreme distribution with most candadites not able to reduce travel times in a statsitically significant way ($\Delta TT = 0$ is normalized to 1)  
We also look at the candadites with the highest in their respective score component

In [ ]:
fig_dist = go.Figure()
for col, label, colour in [
    ("delta_att_norm",      "Travel time reduction",  "#003688"),
    ("net_new_pop_norm",    "Net new demand",         "#00782A"),
    ("deprivation_wt_norm", "Equity / deprivation",   "#9B0056"),
]:
    fig_dist.add_trace(go.Box(
        y=scores2_df[col],
        name=label,
        marker_color=colour,
        boxpoints="outliers",
    ))

fig_dist.update_layout(
    title="Distribution of normalised score components across all candidate edges",
    yaxis_title="Normalised value [0, 1]",
    height=450,
    margin=dict(l=40, r=40, t=60, b=40),
)
fig_dist.show()

# Robustness summary table
overall_winner = winner_counts.most_common(1)[0][0]
print(f"Most robust candidate: {overall_winner}")
print(f"  Wins in {winner_counts[overall_winner]} / {len(combos)} "
      f"({100*winner_counts[overall_winner]/len(combos):.1f}%) of weight combinations\n")

print("Raw component values for top 5 most frequent winners:")
print(f"  {'Candidate':<45} {'ΔTT':>8} {'NewPop':>10} {'DepWt':>7} {'Wins%':>7}")
print("  " + "-" * 80)
for edge, count in winner_counts.most_common(5):
    r   = scores2_df[edge_labels == edge].iloc[0]
    pct = 100 * count / len(combos)
    print(f"  {edge:<45} {r['delta_att']:>8.4f} "
          f"{r['net_new_pop']:>10,.0f} {r['deprivation_wt']:>7.3f} {pct:>7.1f}%")

## 3.4 Limitations

- Other transport methods were not modeled, this is definitely possible but we ran out of time and the complexity as well as the search space would have exploded  
- If other transport methods (including ones like walking and cycling) there would likely be a difference in travel times but this only accounts for people served in the area and journey times from other existing stations
- The 2011 to 2021 LSOA code mismatch means a small fraction of LSOAs have neutral deprivation weights, which slightly understates equity differences at the margin